In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
        # quantization_config=BitsAndBytesConfig(
        #     load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16
        # ),
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [3]:
TRAINING_CACHE_DIR = None if torch.cuda.is_available() else ".training_cache"
VALIDATION_CACHE_DIR = None if torch.cuda.is_available() else ".validation_cache"
NUM_TRAINING_TOKENS = int(1e7) if torch.cuda.is_available() else int(1e6)
EVAL_INTERVAL = int(1e5)
NUM_VALIDATION_TOKENS = int(1e6) if torch.cuda.is_available() else int(1e5)
D_SAE = model.d_model * 4
TOPK = 100
TOKENIZER_BATCH_SIZE = 128
FINETUNE_FRACTION = 0.1
# Note this will use up ~1.8GB of space, set to False if you want to skip
SAVE_FINAL_RESULTS = True

In [4]:
from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

checkpoint_dir = "/workspace/sae_checkpoints/gemma_2_2b/next_layer/"

saes = {}

# Load the latest checkpoints for each layer
for layer in range(model.num_layers - 1, -1, -1):
    checkpoint = find_latest_checkpoint(checkpoint_dir, layer)
    if checkpoint is not None:
        saes[layer] = load_checkpoint(checkpoint).sae
        saes[layer].onload()
        print(f"Loaded checkpoint for layer {layer}")
        break
    else:
        print(f"No checkpoint found for layer {layer}")
        # raise ValueError(f"No checkpoint found for layer {layer}")


Loaded checkpoint for layer 25


In [5]:
import numpy as np

from transformers_sae.validation import run_validations

# validations = run_validations(
#     model,
#     tokenizer,
#     saes,
#     validation_dataset,
#     TOKENIZER_BATCH_SIZE,
#     TRAINING_BATCH_SIZE,
#     NUM_VALIDATION_TOKENS,
#     cache_dir=VALIDATION_CACHE_DIR,
#     start_layer=25
# )
# print(
#     f"mean rre={ {k: np.mean(v.rre).item() for k, v in validations.layer_results.items() if v.rre is not None} }"
# )
# print(
#     f"mean l0={ {k: np.mean(v.l0).item() for k, v in validations.layer_results.items() if v.l0 is not None} }"
# )
# print(
#     f"geom mean kl={ {k: np.exp(np.mean(np.log(np.clip(v.kl, min=1e-9)))).item() for k, v in validations.layer_results.items() if v.kl is not None} })"
# )
# print(
#     f"arith mean kl={ {k: np.mean(v.kl).item() for k, v in validations.layer_results.items() if v.kl is not None} }"
# )
# print(
#     f"live features={ {k: sum(v.live_features) / saes[25].config.d_sae for k, v in validations.layer_results.items() if v.live_features is not None} }"
# )

In [8]:
t = torch.ones((2, 5, 10))

t.view((2 * 5), -1)[torch.tensor([[0,6,7], [3, 5, 8]]), :] = 0

t

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]])

In [23]:
from transformers_sae.validation import generate_with_replacement

# no SAES
# Paris, is a city that is known for its rich history, culture, and architecture.
# It is also a city that is known for its fashion and style.
# Paris is a city that is always changing and evolving, and it is a city that is always on the cutting edge
# of fashion.

with torch.autocast(
    device_type="cuda" if model.device.type == "cuda" else "cpu",
    dtype=torch.bfloat16,
):
    generate_with_replacement(
        model,
        tokenizer,
        "The capital of France,",
        # {25: gemma_scope},
        {},
        # {
        #     layer: sae
        #     for layer, sae in saes.items()
        #     if layer > 24
        # },
    );

 Paris, is a city that is known for its rich history, culture, and architecture. It is also a city that is known for its fashion and style. Paris is a city that is always changing and evolving, and it is a city that is always on the cutting edge of fashion.

Paris is a city that is known for its fashion and style. The city is home to some of the most famous fashion houses in the world, and it is also home to some of the most stylish people


In [12]:
from transformers_sae.sae_lens_wrapper import wrap_sae_lens_pretrained

gemma_scope = wrap_sae_lens_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_25/width_16k/canonical",
    device=TRAINING_DEVICE,
)

layer_25/width_16k/average_l0_116/params(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

In [115]:
from transformers_sae.validation import generate_with_replacement

with torch.autocast(
    device_type="cuda" if model.device.type == "cuda" else "cpu",
    dtype=torch.bfloat16,
):
    generate_with_replacement(
        model,
        tokenizer,
        "The capital of France,",
        # {25: gemma_scope},
        {},
        # {layer: sae for layer, sae in saes.items() if layer > 24},
    );

 Paris, is a city that is known for its rich history, culture, and architecture. It is also a city that is known for its fashion and style. Paris is a city that is always changing and evolving, and it is always a city that is full of surprises.

Paris is a city that is full of surprises. It is a city that is full of surprises. It is a city that is full of surprises. It is a city that is full of surprises. It is a city


Traceback (most recent call last):
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 520, in _process_child_correspondence
    self.evaluate_child(new)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 665, in evaluate_child
    return child.evaluate(self.get_globals(), None)
           ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 285, in evaluate
    exec(code, glb, lcl)
    ~~~~^^^^^^^^^^^^^^^^
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/transformers_sae/encoder.py", line 11, in <adjust>
    EncoderKind: TypeAlias = Literal["relu", "topk", "batch_topk"]
                 ^^^^^^^^^
NameError: name 'TypeAlias' is not defined

